
# 応用問題: MNIST 手書き数字の認識 (行列積 = AI推論)

## 目標

**ニューラルネットワークの推論 (forward) の正体は「行列積 + 活性化関数」** である。これまで何度も並列化してきた行列(ベクトル)積が, そのまま AI の推論計算になっていることを, **本物の MNIST 手書き数字**を認識して体感する。

題材は 2層 MLP:

- 入力 784次元 (28×28 画像を一列に並べたもの) -> 隠れ層 128 (ReLU) -> 出力 10クラス
- `h = ReLU(W1 x + b1)` (128次元), `o = W2 h + b2` (10次元), 予測クラス = `argmax(o)`

`data/` に **学習済みの重み**と **本物のテスト画像** が用意してある:

- `data/mnist_weights.txt` : 学習済みの重み `W1, b1, W2, b2`
- `data/mnist_test.txt` : MNIST テスト画像 1000 枚 (各 784 画素, 0..255) と正解ラベル

(重みは MNIST 訓練データであらかじめ学習させたもの。学習そのものは `02_mlp_train` で扱う。)

## 課題

`mnist_infer.cpp` (または `mnist_infer.f90`) は, 重みと画像を読み込み, 各画像を forward して予測クラスを求め, 正解率を表示する。各画像の推論は互いに**独立**なので並列化できる。

`TODO` の箇所に **OpenMP の指示を追加**し, 画像のループを並列化せよ。

- C++: ループ直前に `#pragma omp parallel for reduction(+:correct)` を加える (正解数を数えるので reduction)。
- Fortran: ループを `!$omp parallel do private(...) reduction(+:correct)` と `!$omp end parallel do` で囲む。

各画像の推論で使う一時配列 (隠れ層 `h`) は各スレッドで別々に持つ必要がある (C++ はループ内で宣言済み, Fortran は `private` 指定)。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mnist_infer.cpp -o mnist_infer.exe

# Fortran
nvfortran -fast -mp=multicore mnist_infer.f90 -o mnist_infer.exe
```

```
OMP_NUM_THREADS=4 ./mnist_infer.exe
```

## 期待される結果

```
MNIST テスト 1000 枚: 正解 979 枚, 正解率 = 97.90%
```

- 学習済みの重みを使うので, **本当に手書き数字を約 98% 当てる**。並列化した行列積がそのまま認識器になっている。
- 各画像の計算は独立かつ決定的なので, **スレッド数を変えても正解率は完全に同じ** (97.90%) になる。
- スレッド数を増やすと速くなる (「性能を比べる」セルで台数効果を測れる)。
- (発展) GPU にオフロードする, 行列積を SIMD 化する, などにも挑戦してみよ。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ mnist_infer.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* 本物の MNIST 手書き数字を, 学習済みの2層MLPで認識する (推論=forward)。
   - data/mnist_weights.txt : 学習済みの重み (784->128->10)
   - data/mnist_test.txt    : テスト画像 (28x28=784画素, 0..255) と正解ラベル
   推論の中身は「行列ベクトル積 + 活性化(ReLU) + argmax」。これまで並列化してきた
   行列計算が, そのまま手書き数字の認識になる。各画像の推論は独立なので並列化できる。 */
int main(int argc, char ** argv) {
  int IN, HID, OUT;

  /* --- 重みの読み込み --- */
  FILE * fw = fopen("data/mnist_weights.txt", "r");
  if (!fw) { printf("data/mnist_weights.txt が開けません\n"); return 1; }
  if (fscanf(fw, "%d %d %d", &IN, &HID, &OUT) != 3) return 1;
  double * W1 = (double *)malloc(sizeof(double) * HID * IN);
  double * b1 = (double *)malloc(sizeof(double) * HID);
  double * W2 = (double *)malloc(sizeof(double) * OUT * HID);
  double * b2 = (double *)malloc(sizeof(double) * OUT);
  for (long k = 0; k < (long)HID * IN; k++)  fscanf(fw, "%lf", &W1[k]);
  for (int k = 0; k < HID; k++)              fscanf(fw, "%lf", &b1[k]);
  for (long k = 0; k < (long)OUT * HID; k++) fscanf(fw, "%lf", &W2[k]);
  for (int k = 0; k < OUT; k++)              fscanf(fw, "%lf", &b2[k]);
  fclose(fw);

  /* --- テスト画像の読み込み (画素 0..255 -> 0..1 に正規化) --- */
  FILE * ft = fopen("data/mnist_test.txt", "r");
  if (!ft) { printf("data/mnist_test.txt が開けません\n"); return 1; }
  int NT, IN2;
  if (fscanf(ft, "%d %d", &NT, &IN2) != 2) return 1;
  double * X = (double *)malloc(sizeof(double) * (long)NT * IN);
  int *    y = (int *)malloc(sizeof(int) * NT);
  for (int i = 0; i < NT; i++) {
    for (int k = 0; k < IN; k++) { int v; fscanf(ft, "%d", &v); X[(long)i*IN+k] = v / 255.0; }
    fscanf(ft, "%d", &y[i]);
  }
  fclose(ft);

  /* --- 推論: 各画像を MLP に通して予測クラス(argmax)を求め, 正解数を数える --- */
  long correct = 0;
  double t0 = omp_get_wtime();
  // TODO: 各画像の推論は独立。#pragma omp parallel for reduction(+:correct) で並列化せよ.
  for (int i = 0; i < NT; i++) {
    double h[1024];                       /* 隠れ層 (HID<=1024 を仮定) */
    const double * x = &X[(long)i * IN];
    for (int hh = 0; hh < HID; hh++) {    /* h = ReLU(W1 x + b1) */
      double s = b1[hh];
      const double * w = &W1[(long)hh * IN];
      for (int k = 0; k < IN; k++) s += w[k] * x[k];
      h[hh] = (s > 0.0) ? s : 0.0;
    }
    int best = 0; double bestv = -1e300;  /* o = W2 h + b2, argmax */
    for (int oo = 0; oo < OUT; oo++) {
      double s = b2[oo];
      const double * w = &W2[(long)oo * HID];
      for (int hh = 0; hh < HID; hh++) s += w[hh] * h[hh];
      if (s > bestv) { bestv = s; best = oo; }
    }
    if (best == y[i]) correct++;
  }
  double elapsed = omp_get_wtime() - t0;

  printf("MNIST テスト %d 枚: 正解 %ld 枚, 正解率 = %.2f%%\n",
         NT, correct, 100.0 * correct / NT);
  printf("elapsed = %.3f sec\n", elapsed);
  free(W1); free(b1); free(W2); free(b2); free(X); free(y);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore mnist_infer.cpp -o mnist_infer_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mnist_infer_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ mnist_infer.f90
! 本物の MNIST 手書き数字を, 学習済みの2層MLPで認識する (推論=forward)。
!   data/mnist_weights.txt : 学習済みの重み (784->128->10)
!   data/mnist_test.txt    : テスト画像 (28x28=784画素, 0..255) と正解ラベル
! 推論の中身は「行列ベクトル積 + 活性化(ReLU) + argmax」。各画像の推論は独立なので並列化できる。
program mnist_infer
  use omp_lib
  implicit none
  integer :: IN, HID, OUT, NT, IN2, i, hh, oo, k, best, lab
  integer(8) :: correct
  real(8) :: s, bestv, t0, elapsed
  real(8), allocatable :: W1(:,:), b1(:), W2(:,:), b2(:), X(:,:), hidv(:)
  integer, allocatable :: y(:), pix(:)

  ! --- 重みの読み込み ---
  open(10, file="data/mnist_weights.txt", status="old", action="read")
  read(10, *) IN, HID, OUT
  allocate(W1(IN,HID), b1(HID), W2(HID,OUT), b2(OUT))   ! W1(k,hh)=W1[hh][k] の並び
  read(10, *) W1        ! HID*IN 個 (hh ごとに IN 個ずつ) を列優先 W1(:,hh) に読み込む
  read(10, *) b1
  read(10, *) W2        ! OUT*HID 個
  read(10, *) b2
  close(10)

  ! --- テスト画像の読み込み (画素 0..255 -> 0..1) ---
  open(10, file="data/mnist_test.txt", status="old", action="read")
  read(10, *) NT, IN2
  allocate(X(IN,NT), y(NT), pix(IN))
  do i = 1, NT
     read(10, *) pix, lab
     X(:,i) = real(pix, 8) / 255.0d0
     y(i) = lab
  end do
  close(10)

  ! --- 推論 ---
  correct = 0
  t0 = omp_get_wtime()
  ! TODO: 各画像の推論は独立。!$omp parallel do reduction(+:correct) で並列化せよ.
  do i = 1, NT
     allocate(hidv(HID))
     do hh = 1, HID                       ! h = ReLU(W1 x + b1)
        s = b1(hh)
        do k = 1, IN
           s = s + W1(k,hh) * X(k,i)
        end do
        hidv(hh) = max(0.0d0, s)
     end do
     best = 1; bestv = -1d300              ! o = W2 h + b2, argmax
     do oo = 1, OUT
        s = b2(oo)
        do hh = 1, HID
           s = s + W2(hh,oo) * hidv(hh)
        end do
        if (s > bestv) then
           bestv = s; best = oo
        end if
     end do
     if (best - 1 == y(i)) correct = correct + 1   ! クラスは 0..9, best は 1..10
     deallocate(hidv)
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  elapsed = omp_get_wtime() - t0

  print "(a,i0,a,i0,a,f0.2,a)", &
       "MNIST テスト ", NT, " 枚: 正解 ", correct, " 枚, 正解率 = ", &
       100.0d0 * correct / NT, " %"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program mnist_infer

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore mnist_infer.f90 -o mnist_infer_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mnist_infer_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:mnist_infer.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mnist_infer.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mnist_infer.cpp}